# Fine-tune Qwen2.5-3B-Instruct bằng DeLoRA trên VlogQA

Notebook này fine-tune **Qwen2.5-3B-Instruct** với **DeLoRA** (Decoupled Low-rank Adaptation):

| Tham số | LoRA chuẩn | **DeLoRA** |
|---|---|---|
| Cơ chế | `ΔW = BA` | Normalize A, B + learnable `λ` boundary + layer-wise scaling |
| Quantization | ✅ QLoRA | ❌ **Không tương thích** — phải dùng full precision |
| Learning Rate | `2e-4` | **`5e-3`** (cao hơn 25x) |
| `lambda_init` | N/A | `10–15` (boundary parameter) |
| Robustness | Nhạy với hyperparams | **Ổn định hơn** |
| Framework | Unsloth / HF PEFT | **HF PEFT** (native) |

**Hardware target:** RTX 3090 (24GB VRAM)  
**Dataset:** VlogQA (Extractive QA tiếng Việt, dạng SQuAD)  
**Evaluation:** Exact Match (EM) và F1-score.

## Bước 0: Cài đặt thư viện

In [ ]:
# Bước 0a: Gỡ cài đặt các gói cũ để tránh xung đột
# DeLoRA KHÔNG cần unsloth — dùng HF Transformers + PEFT thuần
!pip uninstall torch torchvision torchaudio xformers -y 2>/dev/null || true

In [1]:
# Bước 2: Cài đặt lại PyTorch với CUDA 12.8
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 69.9 MB/s  0:00:09:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 76.0 MB/s  0:00:03:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 74.1 MB/s  0:00:03:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 79.6 MB/s  0:00:01ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 66.9 MB/s  0:00:08:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 240.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 81.4 MB/s  0:00:02ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 193.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 89.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 84.7 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 78.4 MB/s 

In [2]:
# Bước 3: Cài đặt transformers, trl, peft, accelerate, bitsandbytes, xformers và unsloth
!pip install transformers trl peft accelerate bitsandbytes xformers
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo
!pip install -q wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 44.1 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 15.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 46.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 41.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 18.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 12.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 53.5 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 34.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 9.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 55.4 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.8/801.8 kB 15.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0

In [12]:
!pip install -q wandb


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [1]:
# Kiểm tra version PEFT (phải >= 0.15.0 để có DeLoRA)
import peft
import torch
print(f"PEFT version : {peft.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to load /opt/venv/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /opt/venv/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /opt/venv/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /opt/venv/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


PEFT version : 0.19.1
PyTorch      : 2.11.0+cu128
CUDA available: True
GPU          : NVIDIA GeForce RTX 3090
VRAM         : 25.30 GB


In [2]:
# ============================================================
# CFG - Toan bo hyperparameters va cau hinh o mot cho duy nhat
# Moi thay doi chi can sua o day, tu dong dong bo voi W&B
# ============================================================
CFG = {
    # --- Model ---
    "model_name"       : "Qwen/Qwen2.5-1.5B-Instruct",

    # --- DeLoRA ---
    "peft_method"      : "delora",
    "r"                : 16,
    "delora_lambda"    : 15,       # Boundary parameter lambda
    "module_dropout"   : 0.05,
    "target_modules"   : ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],

    # --- Training ---
    "learning_rate"              : 5e-3,   # 25x cao hon LoRA
    "lr_scheduler_type"          : "cosine",
    "warmup_ratio"               : 0.03,
    "num_train_epochs"           : 5,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size" : 4,
    "gradient_accumulation_steps": 4,     # Effective batch = 16
    "weight_decay"               : 0.01,
    "optim"                      : "adamw_torch",
    "seed"                       : 3407,

    # --- Sequence ---
    "max_seq_length"             : 8192,
    "max_context_tokens"         : 7500,

    # --- Early Stopping ---
    "early_stopping_patience"    : 5,
    "eval_steps"                 : 100, 

    # --- Paths ---
    "train_path"                 : "train.json",
    "dev_path"                   : "dev.json",
    "test_path"                  : "test.json",
    "save_path"                  : "qwen2.5-1.5b-instruct-delora-vlogqa",
    "checkpoint_dir"             : "outputs_delora_vlogqa",
    "result_file"                : "delora_vlogqa_test_results.json",

    # --- W&B ---
    "wandb_project"              : "PACLIC_2026-VlogQA",
    "wandb_run_name"             : "qwen2.5-1.5b-delora-vlogqa",
    "wandb_tags"                 : ["delora", "qwen2.5-3b", "vlogqa", "paclic2026"],
}

print("CFG initialized:")
for k, v in CFG.items():
    print(f"  {k:30s}: {v}")

CFG initialized:
  model_name                    : Qwen/Qwen2.5-1.5B-Instruct
  peft_method                   : delora
  r                             : 16
  delora_lambda                 : 15
  module_dropout                : 0.05
  target_modules                : ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  learning_rate                 : 0.005
  lr_scheduler_type             : cosine
  warmup_ratio                  : 0.03
  num_train_epochs              : 5
  per_device_train_batch_size   : 4
  per_device_eval_batch_size    : 4
  gradient_accumulation_steps   : 4
  weight_decay                  : 0.01
  optim                         : adamw_torch
  seed                          : 3407
  max_seq_length                : 8192
  max_context_tokens            : 7500
  early_stopping_patience       : 5
  eval_steps                    : 100
  train_path                    : train.json
  dev_path                      : dev.json
  test_path                 

In [3]:
# ==========================================
# CAI DAT VA KHOI TAO WEIGHTS & BIASES (W&B)
# ==========================================
import wandb
import os

wandb.login()  # Se hoi API key neu chua login

# Truyen thang CFG vao wandb.init -> luon dong bo, khong bao gio lech
wandb_run = wandb.init(
    project = CFG["wandb_project"],
    name    = CFG["wandb_run_name"],
    config  = CFG,   # <-- toan bo CFG, single source of truth
    tags    = CFG["wandb_tags"],
)

print(f"[W&B] Run initialized: {wandb_run.url}")
print(f"  Project : {CFG['wandb_project']}")
print(f"  Run name: {CFG['wandb_run_name']}")

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: longvuongbrvt (longvuongbrvt-csc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[W&B] Run initialized: https://wandb.ai/longvuongbrvt-csc/PACLIC_2026-VlogQA/runs/hljv599r
  Project : PACLIC_2026-VlogQA
  Run name: qwen2.5-1.5b-delora-vlogqa


## 1. Load Model và Tokenizer (Full BF16 — Không QLoRA)

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("Dang tai tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    CFG["model_name"],
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Tu dong detect FlashAttention2
try:
    import importlib
    importlib.import_module("flash_attn")
    _attn_impl = "flash_attention_2"
    print("FlashAttention2 kha dung - dung flash_attention_2")
except ImportError:
    _attn_impl = "sdpa"  # Scaled Dot-Product Attention (PyTorch 2.x built-in)
    print("FlashAttention2 chua cai - fallback sang sdpa")

print(f"Dang tai model {CFG['model_name']} (BF16, khong quantize)...")
model = AutoModelForCausalLM.from_pretrained(
    CFG["model_name"],
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation=_attn_impl,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f"\nModel da load thanh cong!")
print(f"  Dtype  : {next(model.parameters()).dtype}")
print(f"  Params : {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"  Attn   : {_attn_impl}")
print(f"  VRAM   : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Dang tai tokenizer...


FlashAttention2 chua cai - fallback sang sdpa
Dang tai model Qwen/Qwen2.5-1.5B-Instruct (BF16, khong quantize)...


`torch_dtype` is deprecated! Use `dtype` instead!




Model da load thanh cong!
  Dtype  : torch.bfloat16
  Params : 1.54B
  Attn   : sdpa
  VRAM   : 3.09 GB


## 2. Cấu hình DeLoRA Adapter

**Điểm khác biệt so với LoRA:**
- `DeloraConfig` thay vì `LoraConfig`
- `learning_rate` phải cao hơn **10–50x** (dùng `5e-3`)
- `lambda_init` ~ 10–15 (boundary parameter của DeLoRA)
- `lora_dropout = 0.0` (DeLoRA hoạt động tốt nhất không dropout)
- KHÔNG dùng `quantization_config`

In [5]:
from peft import get_peft_model, DeloraConfig

delora_config = DeloraConfig(
    r              = CFG["r"],
    delora_lambda  = CFG["delora_lambda"],
    target_modules = CFG["target_modules"],
    module_dropout = CFG["module_dropout"],
    bias           = "none",
    task_type      = "CAUSAL_LM",
    init_weights   = True,
)

model = get_peft_model(model, delora_config)
model.enable_input_require_grads()
model.gradient_checkpointing_enable()

print("Cau hinh DeLoRA thanh cong!")
model.print_trainable_parameters()

Cau hinh DeLoRA thanh cong!
trainable params: 18,464,964 || all params: 1,562,179,268 || trainable%: 1.1820


## 3. Chuẩn bị Dataset VlogQA

In [6]:
import json
from datasets import Dataset

def load_vlogqa(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                question = qa["question"]
                answer = qa["answers"][0]["text"] if qa["answers"] else ""
                if answer:
                    samples.append({"context": context, "question": question,
                                    "answer": answer, "id": qa.get("id", "")})
    return samples


def truncate_context_around_answer(context, answer, tokenizer,
                                   max_ctx_tokens=CFG["max_context_tokens"]):
    answer_start = context.find(answer)
    if answer_start == -1:
        ctx_ids = tokenizer.encode(context, add_special_tokens=False)
        return context if len(ctx_ids) <= max_ctx_tokens else \
               tokenizer.decode(ctx_ids[:max_ctx_tokens], skip_special_tokens=True)
    before_ids = tokenizer.encode(context[:answer_start], add_special_tokens=False)
    answer_ids = tokenizer.encode(answer, add_special_tokens=False)
    after_ids  = tokenizer.encode(context[answer_start + len(answer):], add_special_tokens=False)
    if len(before_ids) + len(answer_ids) + len(after_ids) <= max_ctx_tokens:
        return context
    budget = max_ctx_tokens - len(answer_ids)
    half   = budget // 2
    return tokenizer.decode(before_ids[-half:] + answer_ids + after_ids[:budget - half],
                            skip_special_tokens=True)


SYSTEM_PROMPT = (
    "Ban la he thong trich xuat cau tra loi tu van ban tieng Viet. "
    "QUY TAC BAT BUOC:\n"
    "1) Chi tra ve DUNG cum tu xuat hien nguyen van trong doan van.\n"
    "2) Khong viet cau hoan chinh, khong giai thich, khong them tien to nao.\n"
    "3) Cau tra loi phai CO MAT trong doan van."
)
USER_PROMPT_TEMPLATE = (
    "Trich xuat cau tra loi tu doan van. Chi tra ve cum tu trong doan van.\n\n"
    "Doan van:\n{context}\n\nCau hoi: {question}\n\nCau tra loi (span-only):"
)


def format_prompt_train(context, question, answer, tokenizer):
    context_cropped = truncate_context_around_answer(context, answer, tokenizer)
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": USER_PROMPT_TEMPLATE.format(
            context=context_cropped, question=question)},
        {"role": "assistant", "content": answer},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def format_prompt_inference(context, question, tokenizer):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_PROMPT_TEMPLATE.format(
            context=context[:CFG["max_context_tokens"]], question=question)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("Dinh nghia ham xong!")

Dinh nghia ham xong!


In [7]:
from datasets import Dataset

train_samples = load_vlogqa(CFG["train_path"])
print(f"Train: {len(train_samples)} mau")

train_texts = [format_prompt_train(s["context"], s["question"], s["answer"], tokenizer)
               for s in train_samples]
dataset = Dataset.from_dict({"text": train_texts})

dev_samples = load_vlogqa(CFG["dev_path"])
print(f"Dev  : {len(dev_samples)} mau")

dev_texts = [format_prompt_train(s["context"], s["question"], s["answer"], tokenizer)
             for s in dev_samples]
eval_dataset = Dataset.from_dict({"text": dev_texts})

print(f"\nVi du prompt (150 ky tu dau): {train_texts[0][:150]}...")

Train: 8386 mau
Dev  : 999 mau

Vi du prompt (150 ky tu dau): <|im_start|>system
Ban la he thong trich xuat cau tra loi tu van ban tieng Viet. QUY TAC BAT BUOC:
1) Chi tra ve DUNG cum tu xuat hien nguyen van tron...


## 4. Cấu hình và Bắt đầu Huấn luyện

**Lưu ý quan trọng về DeLoRA hyperparameters:**
- `learning_rate = 5e-3` — cao hơn LoRA thường ~25x
- `optim = "adamw_torch"` — DeLoRA hoạt động tốt nhất với AdamW chuẩn (không phải 8-bit)
- `per_device_train_batch_size = 4` — RTX 3090 24GB có thể chịu được batch lớn hơn
- `gradient_accumulation_steps = 4` — Effective batch = 16
- `lr_scheduler_type = "cosine"` — Quan trọng với LR cao của DeLoRA

In [13]:
# ============================================================
# TRL 0.24.0 API:
#   - Dung SFTConfig (ke thua TrainingArguments), KHONG dung TrainingArguments truc tiep
#   - max_seq_length da bi xoa khoi SFTConfig -> set qua tokenizer.model_max_length
#   - packing=True van con trong SFTConfig
#   - dataset_text_field con trong SFTConfig (neu co), fallback sang formatting_func
#   - SFTTrainer chi nhan: model, args, datasets, processing_class, formatting_func, callbacks
# ============================================================
import inspect
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
import sys

# Set max sequence length qua tokenizer (cach dung trong TRL >= 0.12)
tokenizer.model_max_length = CFG["max_seq_length"]
print(f"tokenizer.model_max_length = {tokenizer.model_max_length}")

# Kiem tra nhung params nao SFTConfig chap nhan
sft_cfg_params = set(inspect.signature(SFTConfig.__init__).parameters.keys())
print(f"SFTConfig has packing          : {'packing' in sft_cfg_params}")
print(f"SFTConfig has dataset_text_field: {'dataset_text_field' in sft_cfg_params}")
print(f"SFTConfig has max_seq_length   : {'max_seq_length' in sft_cfg_params}")

# ============================================================
# SFTConfig: TrainingArguments + SFT-specific params
# ============================================================
sft_cfg_kwargs = dict(
    # --- Batch ---
    per_device_train_batch_size  = CFG["per_device_train_batch_size"],
    per_device_eval_batch_size   = CFG["per_device_eval_batch_size"],
    gradient_accumulation_steps  = CFG["gradient_accumulation_steps"],

    # --- DeLoRA LR ---
    learning_rate                = CFG["learning_rate"],
    lr_scheduler_type            = CFG["lr_scheduler_type"],
    warmup_ratio                 = CFG["warmup_ratio"],

    # --- Epochs ---
    num_train_epochs             = CFG["num_train_epochs"],

    # --- Precision ---
    bf16                         = True,
    fp16                         = False,

    # --- Optimizer ---
    optim                        = CFG["optim"],
    weight_decay                 = CFG["weight_decay"],

    # --- Gradient Checkpointing ---
    gradient_checkpointing       = True,
    gradient_checkpointing_kwargs= {"use_reentrant": False},

    # --- W&B Logging ---
    logging_steps                = 20,
    report_to                    = "wandb",
    run_name                     = CFG["wandb_run_name"],

    # --- Evaluation & Checkpoint ---
    # --- Evaluation & Checkpoint ---
    eval_strategy                = "steps",
    save_strategy                = "steps",
    eval_steps                   = CFG["eval_steps"],
    save_steps                   = CFG["eval_steps"],
    load_best_model_at_end       = True,
    metric_for_best_model        = "eval_loss",
    greater_is_better            = False,
    save_total_limit             = 2,

    # --- Output ---
    output_dir                   = CFG["checkpoint_dir"],
    seed                         = CFG["seed"],
    dataloader_num_workers       = 0,
)

# Them packing neu SFTConfig ho tro (TRL 0.24 co)
if "packing" in sft_cfg_params:
    sft_cfg_kwargs["packing"] = True

# Them dataset_text_field neu SFTConfig ho tro
if "dataset_text_field" in sft_cfg_params:
    sft_cfg_kwargs["dataset_text_field"] = "text"

training_args = SFTConfig(**sft_cfg_kwargs)

# ============================================================
# formatting_func: fallback neu dataset_text_field khong co
# ============================================================
sft_trainer_params = set(inspect.signature(SFTTrainer.__init__).parameters.keys())

trainer_kwargs = dict(
    model         = model,
    args          = training_args,
    train_dataset = dataset,
    eval_dataset  = eval_dataset,
    processing_class = tokenizer,
    callbacks     = [EarlyStoppingCallback(
        early_stopping_patience=CFG["early_stopping_patience"])],
)

# Neu dataset_text_field khong co trong SFTConfig -> dung formatting_func
if "dataset_text_field" not in sft_cfg_params and "formatting_func" in sft_trainer_params:
    trainer_kwargs["formatting_func"] = lambda x: x["text"]
    print("[Info] Dung formatting_func (TRL 0.24+ khong co dataset_text_field trong SFTConfig)")

trainer = SFTTrainer(**trainer_kwargs)

# Fix sys.modules de tranh loi pickle
real_config_cls  = type(trainer.args)
real_trainer_cls = type(trainer)
sys.modules['trl.trainer.sft_config']  = sys.modules[real_config_cls.__module__]
sys.modules['trl.trainer.sft_trainer'] = sys.modules[real_trainer_cls.__module__]
sys.modules[real_config_cls.__module__].SFTConfig   = real_config_cls
sys.modules[real_trainer_cls.__module__].SFTTrainer = real_trainer_cls

eff_batch = CFG["per_device_train_batch_size"] * CFG["gradient_accumulation_steps"]
print(f"\nTrainer ready! (TRL {__import__('trl').__version__})")
print(f"  Effective batch  : {eff_batch}")
print(f"  Learning rate    : {CFG['learning_rate']}")
print(f"  Max seq length   : {tokenizer.model_max_length} (set via tokenizer)")
print(f"  Packing          : {getattr(training_args, 'packing', 'N/A')}")
print(f"  W&B run          : {wandb_run.url}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[RANK 0] Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
[RANK 0] You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flas

tokenizer.model_max_length = 8192
SFTConfig has packing          : True
SFTConfig has dataset_text_field: True
SFTConfig has max_seq_length   : False



Trainer ready! (TRL 0.24.0)
  Effective batch  : 16
  Learning rate    : 0.005
  Max seq length   : 8192 (set via tokenizer)
  Packing          : True
  W&B run          : https://wandb.ai/longvuongbrvt-csc/PACLIC_2026-VlogQA/runs/zqq454ju


In [14]:
import torch

print(f"VRAM truoc train: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"\nBat dau fine-tuning DeLoRA (patience={CFG['early_stopping_patience']}, max {CFG['num_train_epochs']} epochs)...\n")

trainer_stats = trainer.train()

print(f"\nHoan tat Training!")
print(f"  Epochs da chay  : {trainer_stats.metrics.get('epoch', 'N/A'):.2f}")
print(f"  Train Loss cuoi : {trainer_stats.metrics.get('train_loss', 'N/A'):.4f}")
print(f"  VRAM peak       : {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

wandb.finish()
print(f"[W&B] Run finalized: {wandb_run.url}")

VRAM truoc train: 6.71 GB

Bat dau fine-tuning DeLoRA (patience=5, max 5 epochs)...



Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.834666,2.924999,2.235340,1637096.000000,0.464463
200,1.434131,3.151317,2.014508,3273436.000000,0.455756
300,1.115532,3.478809,1.661036,4909866.000000,0.448564
400,0.886516,3.574732,1.696327,6546649.000000,0.442567
500,0.664501,3.772775,1.623427,8183226.000000,0.436401
600,0.326145,4.124065,1.400587,9818953.000000,0.433649



Hoan tat Training!
  Epochs da chay  : 1.15
  Train Loss cuoi : 1.1551
  VRAM peak       : 15.07 GB


eval/entropy,█▆▃▃▃▁
eval/loss,▁▂▄▅▆█
eval/mean_token_accuracy,█▆▄▃▂▁
eval/num_tokens,▁▂▄▅▇█
eval/runtime,▁▁█▄▅▆
eval/samples_per_second,██▁▆▄▃
eval/steps_per_second,██▁█▁▁
train/entropy,█▅▅▅▅▄▄▄▄▄▃▄▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁
train/epoch,▁▁▁▂▂▂▂▃▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▃▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇████
+5,...


[W&B] Run finalized: https://wandb.ai/longvuongbrvt-csc/PACLIC_2026-VlogQA/runs/zqq454ju


## 5. Lưu Model

In [16]:
model.save_pretrained(CFG["save_path"])
tokenizer.save_pretrained(CFG["save_path"])

print(f"Da luu mo hinh DeLoRA tai: {CFG['save_path']}")
import os
for f in sorted(os.listdir(CFG["save_path"])):
    size = os.path.getsize(os.path.join(CFG["save_path"], f))
    print(f"  {f:40s} {size/1e6:.1f} MB")

Da luu mo hinh DeLoRA tai: qwen2.5-3b-instruct-delora-vlogqa
  README.md                                0.0 MB
  adapter_config.json                      0.0 MB
  adapter_model.safetensors                123.2 MB
  chat_template.jinja                      0.0 MB
  tokenizer.json                           11.4 MB
  tokenizer_config.json                    0.0 MB


## 6. Kiểm tra Inference nhanh sau khi huấn luyện

In [17]:
import torch

model.eval()
model.config.use_cache = True

sample = train_samples[0]
prompt = format_prompt_inference(sample["context"], sample["question"], tokenizer)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens      = 64,
        temperature         = 0.01,
        do_sample           = False,
        repetition_penalty  = 1.1,
        eos_token_id        = tokenizer.eos_token_id,
        pad_token_id        = tokenizer.pad_token_id,
    )

generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print("\n--- KẾT QUẢ INFERENCE ---")
print(f"Câu hỏi      : {sample['question']}")
print(f"Đáp án đúng  : {sample['answer']}")
print(f"Model trả lời: {response}")
print(f"Exact Match  : {response.strip() == sample['answer'].strip()}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- KẾT QUẢ INFERENCE ---
Câu hỏi      : Nước tương được sử dụng để nêm hãng gì?
Đáp án đúng  : Maggi
Model trả lời: muối
Exact Match  : False


## 7. Đánh giá đầy đủ trên tập Test (EM & F1)

## 8. (Optional) Load lại adapter để inference sau này

In [2]:
# ==========================================
# CELL NAY CO THE CHAY DOC LAP SAU KHI RESTART KERNEL
# (Chi can ban da chay cell CFG o Buoc 0.5 truoc do)
# ==========================================
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
import importlib

# Tu dong detect FlashAttention2 de tang toc do inference
try:
    importlib.import_module("flash_attn")
    _attn_impl = "flash_attention_2"
    print("FlashAttention2 kha dung - bat flash_attention_2 de tang toc!")
except ImportError:
    _attn_impl = "sdpa"
    print("FlashAttention2 chua cai - dung sdpa")

print("Dang load base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    CFG["model_name"], torch_dtype=torch.bfloat16,
    device_map="auto", trust_remote_code=True,
    attn_implementation=_attn_impl,
)
print("Dang load va merge DeLoRA adapter de tang toc do generate...")
model_loaded = PeftModel.from_pretrained(base_model, CFG["save_path"])
# Merge adapter vao base model luon -> Inference cuc nhanh, khong bi overhead cua PEFT
model_loaded = model_loaded.merge_and_unload()

tokenizer_loaded = AutoTokenizer.from_pretrained(CFG["save_path"])

print(f"Load va Merge thanh cong adapter tu: {CFG['save_path']}")

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to load /opt/venv/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /opt/venv/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /opt/venv/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /opt/venv/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


FlashAttention2 chua cai - dung sdpa
Dang load base model...


`torch_dtype` is deprecated! Use `dtype` instead!



Dang load va merge DeLoRA adapter de tang toc do generate...
Load va Merge thanh cong adapter tu: qwen2.5-3b-instruct-delora-vlogqa


In [7]:
# ==========================================
# CELL NAY HOAN TOAN DOC LAP (CHUA FULL DATA & METRICS)
# ==========================================
import json, re, string
import torch
from tqdm import tqdm

# --- 1. Dinh nghia lai cac ham can thiet de chay doc lap ---
def load_vlogqa(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                question = qa["question"]
                answer = qa["answers"][0]["text"] if qa["answers"] else ""
                if answer:
                    samples.append({"context": context, "question": question,
                                    "answer": answer, "id": qa.get("id", "")})
    return samples

SYSTEM_PROMPT = (
    "Ban la he thong trich xuat cau tra loi tu van ban tieng Viet. "
    "QUY TAC BAT BUOC:\n"
    "1) Chi tra ve DUNG cum tu xuat hien nguyen van trong doan van.\n"
    "2) Khong viet cau hoan chinh, khong giai thich, khong them tien to nao.\n"
    "3) Cau tra loi phai CO MAT trong doan van."
)
USER_PROMPT_TEMPLATE = (
    "Trich xuat cau tra loi tu doan van. Chi tra ve cum tu trong doan van.\n\n"
    "Doan van:\n{context}\n\nCau hoi: {question}\n\nCau tra loi (span-only):"
)

def format_prompt_inference(context, question, tokenizer):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_PROMPT_TEMPLATE.format(
            context=context[:CFG["max_context_tokens"]], question=question)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def normalize_answer(s):
    s = s.lower()
    s = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', s)
    return ' '.join(s.split())

def compute_exact_match(pred, gold):
    return int(normalize_answer(pred) == normalize_answer(gold))

def compute_f1(pred, gold):
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    common = set(pred_tokens) & set(gold_tokens)
    if not common: return 0.0
    precision = len(common) / len(pred_tokens)
    recall    = len(common) / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

# --- 2. Load Data ---
test_samples = load_vlogqa(CFG["test_path"])
print(f"Test set: {len(test_samples)} mau")

# --- 3. Run Inference ---
model_loaded.eval()
model_loaded.config.use_cache = True

tokenizer_loaded.padding_side = "left"
if tokenizer_loaded.pad_token is None:
    tokenizer_loaded.pad_token = tokenizer_loaded.eos_token

loaded_results, loaded_em_scores, loaded_f1_scores = [], [], []
BATCH_SIZE = 16  # Tang tu 8 len 16 de day nhanh toc do

for i in tqdm(range(0, len(test_samples), BATCH_SIZE), desc="Evaluating Loaded Model (Batched)"):
    batch_samples = test_samples[i : i + BATCH_SIZE]
    prompts = [
        format_prompt_inference(s["context"], s["question"], tokenizer_loaded)
        for s in batch_samples
    ]
    
    inputs = tokenizer_loaded(
        prompts, return_tensors="pt", padding=True, 
        truncation=True, max_length=CFG["max_seq_length"]
    ).to("cuda")
    
    with torch.no_grad():
        outputs = model_loaded.generate(
            **inputs, max_new_tokens=64, do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer_loaded.eos_token_id,
            pad_token_id=tokenizer_loaded.pad_token_id,
        )
        
    for j, s in enumerate(batch_samples):
        input_len = inputs["input_ids"][j].shape[-1]
        generated_ids = outputs[j][input_len:]
        prediction = tokenizer_loaded.decode(generated_ids, skip_special_tokens=True).strip()
        
        em = compute_exact_match(prediction, s["answer"])
        f1 = compute_f1(prediction, s["answer"])
        loaded_em_scores.append(em)
        loaded_f1_scores.append(f1)
        loaded_results.append({"id": s["id"], "question": s["question"],
                        "gold": s["answer"], "prediction": prediction, "em": em, "f1": f1})
        
        match_status = "✅" if em == 1 else "❌"
        print(f"[{i+j+1}/{len(test_samples)}] {match_status} Gold: '{s['answer']}' | Pred: '{prediction}'")

tokenizer_loaded.padding_side = "right"

avg_loaded_em = sum(loaded_em_scores) / len(loaded_em_scores) * 100
avg_loaded_f1 = sum(loaded_f1_scores) / len(loaded_f1_scores) * 100

print(f"\n{'='*50}")
print(f"  Exact Match (EM) tren loaded model: {avg_loaded_em:.2f}%")
print(f"  F1 Score tren loaded model        : {avg_loaded_f1:.2f}%")
print(f"{'='*50}")

LOADED_RESULT_FILE = "delora_vlogqa_loaded_model_test_results.json"
with open(LOADED_RESULT_FILE, "w", encoding="utf-8") as f:
    json.dump({
        "model": CFG["model_name"], "adapter": CFG["save_path"],
        "peft_method": "DeLoRA (Loaded Model)",
        "exact_match": avg_loaded_em, "f1": avg_loaded_f1,
        "num_test_samples": len(test_samples), "predictions": loaded_results,
    }, f, ensure_ascii=False, indent=2)

print(f"\nDa luu ket qua cua loaded model tai: {LOADED_RESULT_FILE}")


Test set: 1062 mau
[1/1062] ✅ Gold: '10 phút' | Pred: '10 phút'
[2/1062] ❌ Gold: 'mấy cái hoa hồi' | Pred: 'hoa hoa hồi cho dễ thương hơn [âm nhạc] khi cắt bánh ra thì mình thấy bánh có rễ tre Khắc Thông bánh nè [âm nhạc] [Vỗ tay] [âm nhạc] [Vỗ tay] [âm nhạc] [Vỗ tay] [âm nhạc]'
[3/1062] ❌ Gold: 'nước ấm' | Pred: '300ml nước cốt dừa với sữa đặc vào cối giã nhuyễn ra để cho ra hết cái vị ngọt thơm của dừa và sữa đặc đó các bạn ạ [Vỗ tay] [Vỗ tay] [Vỗ tay] [Vỗ tay'
[4/1062] ❌ Gold: 'tạo rễ tre cho bánh và để bánh được chín đều' | Pred: 'Tạo rèn cho bánh và để bánh được chín đều hơn khi mình nướng bánh ở nhiệt độ 175 độ C 350 độ F á [Vỗ tay] [Vỗ tay] [Vỗ tay] [Vỗ tay] [Vỗ tay'
[5/1062] ✅ Gold: 'nước cốt dừa' | Pred: 'nước cốt dừa'
[6/1062] ❌ Gold: 'ngon hơn' | Pred: 'trứng gà thì không có bị "rụng" nhưng mà trứng vịt thì có khả năng rụng hơn trứng gà đó là lý do mình khuyên thường dùng trứng gà để làm bánh hơn là trứng vịt bởi vì trứng vịt khó làm hơn là trứng gà ở nhà mình có thể dùng tr

In [8]:
# ==========================================
# CELL NAY HOAN TOAN DOC LAP (CHUA FULL DATA & METRICS)
# ==========================================
import json, re, string, time
import unicodedata
from collections import Counter
import torch
from tqdm.auto import tqdm

# --- 1. Dinh nghia lai cac ham can thiet de chay doc lap ---
def load_vlogqa(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                question = qa["question"]
                answer = qa["answers"][0]["text"] if qa["answers"] else ""
                if answer:
                    samples.append({"context": context, "question": question,
                                    "answer": answer, "id": qa.get("id", "")})
    return samples

SYSTEM_PROMPT = (
    "Ban la he thong trich xuat cau tra loi tu van ban tieng Viet. "
    "QUY TAC BAT BUOC:\n"
    "1) Chi tra ve DUNG cum tu xuat hien nguyen van trong doan van.\n"
    "2) Khong viet cau hoan chinh, khong giai thich, khong them tien to nao.\n"
    "3) Cau tra loi phai CO MAT trong doan van."
)
USER_PROMPT_TEMPLATE = (
    "Trich xuat cau tra loi tu doan van. Chi tra ve cum tu trong doan van.\n\n"
    "Doan van:\n{context}\n\nCau hoi: {question}\n\nCau tra loi (span-only):"
)

PREFIX_RE = re.compile(
    r"^(đáp án|answer|câu trả lời|theo đoạn văn|trong đoạn văn|trả lời|span-only)\s*[:\-]?\s*",
    re.IGNORECASE,
)

def format_prompt_inference(context, question, tokenizer):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_PROMPT_TEMPLATE.format(
            context=context[:CFG["max_context_tokens"]], question=question)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text or "")
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())

def compute_exact_match(prediction: str, ground_truth: str) -> int:
    return int(normalize_text(prediction) == normalize_text(ground_truth))

def compute_f1_token(prediction: str, ground_truth: str) -> float:
    pred_tokens = normalize_text(prediction).split()
    true_tokens = normalize_text(ground_truth).split()
    if len(pred_tokens) == 0 and len(true_tokens) == 0: return 1.0
    if len(pred_tokens) == 0 or len(true_tokens) == 0: return 0.0
    common = Counter(pred_tokens) & Counter(true_tokens)
    num_common = sum(common.values())
    if num_common == 0: return 0.0
    precision = num_common / len(pred_tokens)
    recall    = num_common / len(true_tokens)
    return (2 * precision * recall) / (precision + recall)

def clean_prediction(raw: str) -> str:
    pred = raw.strip().split("\n")[0].strip().strip('"\'\' ')
    return PREFIX_RE.sub("", pred).strip()

def find_best_span(prediction: str, context: str) -> str:
    pred_norm = normalize_text(prediction)
    if not pred_norm: return prediction
    pred_words = pred_norm.split()
    ctx_words  = context.split()
    n = len(pred_words)
    if n == 0 or len(ctx_words) == 0: return prediction
    best_f1   = compute_f1_token(prediction, context)
    best_span = prediction
    min_w = max(1, n // 2)
    max_w = min(2 * n + 5, len(ctx_words))
    for w in range(min_w, max_w + 1):
        for start in range(len(ctx_words) - w + 1):
            span_orig = " ".join(ctx_words[start:start + w])
            f1 = compute_f1_token(prediction, span_orig)
            if f1 > best_f1:
                best_f1   = f1
                best_span = span_orig
    return best_span if best_f1 >= 0.5 else prediction

# --- 2. Load Data ---
test_samples = load_vlogqa(CFG["test_path"])
TOTAL = len(test_samples)
print(f"Test set: {TOTAL} mau")

# --- 3. Run Inference ---
model_loaded.eval()
model_loaded.config.use_cache = True

tokenizer_loaded.padding_side = "left"
if tokenizer_loaded.pad_token is None:
    tokenizer_loaded.pad_token = tokenizer_loaded.eos_token

all_em_raw, all_f1_raw, all_em_span, all_f1_span, all_predictions = [], [], [], [], []
BATCH_SIZE = 16
SUMMARY_EVERY = 100
n_batches = (TOTAL + BATCH_SIZE - 1) // BATCH_SIZE
start_time = time.time()

print(f"\nBat dau danh gia tren {TOTAL} cau hoi...")
print(f"Batch size = {BATCH_SIZE} | So batch = {n_batches}\n")

pbar = tqdm(range(0, TOTAL, BATCH_SIZE), desc="Evaluating", total=n_batches)

for batch_start in pbar:
    batch = test_samples[batch_start : batch_start + BATCH_SIZE]
    prompts = [
        format_prompt_inference(s["context"], s["question"], tokenizer_loaded)
        for s in batch
    ]
    
    inputs = tokenizer_loaded(
        prompts, return_tensors="pt", padding=True, 
        truncation=True, max_length=CFG["max_seq_length"]
    ).to("cuda")
    
    total_input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model_loaded.generate(
            **inputs, max_new_tokens=64, do_sample=False,
            repetition_penalty=1.1, use_cache=True,
            eos_token_id=tokenizer_loaded.eos_token_id,
            pad_token_id=tokenizer_loaded.pad_token_id,
        )
        
    for j, sample in enumerate(batch):
        gen_tokens = outputs[j][total_input_len:]
        raw_prediction = tokenizer_loaded.decode(gen_tokens, skip_special_tokens=True).strip()
        raw_prediction = raw_prediction.split("\n")[0].strip()
        
        cleaned_prediction = clean_prediction(raw_prediction)
        span_prediction    = find_best_span(cleaned_prediction, sample["context"])
        
        # Tinh metric voi 1 answer (string) tu data format
        truth = sample["answer"]
        em_raw  = compute_exact_match(cleaned_prediction, truth)
        f1_raw  = compute_f1_token(cleaned_prediction, truth)
        em_span = compute_exact_match(span_prediction, truth)
        f1_span = compute_f1_token(span_prediction, truth)
        
        all_em_raw.append(em_raw);   all_f1_raw.append(f1_raw)
        all_em_span.append(em_span); all_f1_span.append(f1_span)
        all_predictions.append({
            "id":               sample["id"],
            "question":         sample["question"],
            "ground_truth":     truth,
            "raw_prediction":   raw_prediction,
            "clean_prediction": cleaned_prediction,
            "span_prediction":  span_prediction,
            "em_raw": em_raw, "f1_raw": f1_raw,
            "em_span": em_span, "f1_span": f1_span,
        })
        
        idx = len(all_predictions)
        
        # Log tung cau
        tqdm.write(
            f"[{idx}/{TOTAL}] "
            f"Q: {sample['question'][:40]:<40} | "
            f"Truth: {truth[:30]:<30} | "
            f"Raw: {cleaned_prediction[:30]:<30} | "
            f"Span: {span_prediction[:30]:<30} | "
            f"EM_raw={em_raw} EM_span={em_span} "
            f"F1_raw={f1_raw:.3f} F1_span={f1_span:.3f}"
        )

    # Cap nhat pbar
    idx = len(all_predictions)
    elapsed = time.time() - start_time
    eta = (elapsed / idx) * (TOTAL - idx) if idx > 0 else 0
    cur_em = sum(all_em_span) / idx * 100
    cur_f1 = sum(all_f1_span) / idx * 100
    pbar.set_postfix({"EM_span": f"{cur_em:.1f}%", "F1_span": f"{cur_f1:.1f}%", "ETA": f"{eta/60:.1f}m"})

    # Checkpoint
    if idx % SUMMARY_EVERY < BATCH_SIZE and idx >= SUMMARY_EVERY:
        tqdm.write(f"\n{'='*60}")
        tqdm.write(f"  [Checkpoint ~{idx}/{TOTAL}] Thoi gian: {elapsed:.0f}s | ETA ~{eta/60:.1f} phut")
        tqdm.write(f"  EM  (raw):  {sum(all_em_raw)/idx*100:.2f}%  |  EM  (span): {cur_em:.2f}%")
        tqdm.write(f"  F1  (raw):  {sum(all_f1_raw)/idx*100:.2f}%  |  F1  (span): {cur_f1:.2f}%")
        tqdm.write(f"{'='*60}\n")

tokenizer_loaded.padding_side = "right"
total_time = time.time() - start_time

avg_em_raw = sum(all_em_raw) / TOTAL * 100
avg_f1_raw = sum(all_f1_raw) / TOTAL * 100
avg_em_span = sum(all_em_span) / TOTAL * 100
avg_f1_span = sum(all_f1_span) / TOTAL * 100

print(f"\nHoan tat! Tong thoi gian: {total_time/60:.1f} phut")
print(f"Toc do trung binh: {TOTAL/total_time:.1f} cau/giay")
print(f"\n{'='*50}")
print(f"  EM (raw)  : {avg_em_raw:.2f}%  |  EM (span)  : {avg_em_span:.2f}%")
print(f"  F1 (raw)  : {avg_f1_raw:.2f}%  |  F1 (span)  : {avg_f1_span:.2f}%")
print(f"{'='*50}")

LOADED_RESULT_FILE = "delora_vlogqa_loaded_model_test_results.json"
with open(LOADED_RESULT_FILE, "w", encoding="utf-8") as f:
    json.dump({
        "model": CFG["model_name"], "adapter": CFG["save_path"],
        "peft_method": "DeLoRA (Loaded Model)",
        "metrics_raw":  {"em": avg_em_raw,  "f1": avg_f1_raw},
        "metrics_span": {"em": avg_em_span, "f1": avg_f1_span},
        "num_test_samples": TOTAL, "predictions": all_predictions,
    }, f, ensure_ascii=False, indent=2)

print(f"\nDa luu ket qua cua loaded model tai: {LOADED_RESULT_FILE}")


Test set: 1062 mau

Bat dau danh gia tren 1062 cau hoi...
Batch size = 16 | So batch = 67

[1/1062] Q: Lò được làm nóng trước khi nướng bao lâu | Truth: 10 phút                        | Raw: 10 phút                        | Span: 10 phút                        | EM_raw=1 EM_span=1 F1_raw=1.000 F1_span=1.000
[2/1062] Q: Đầu bếp sử dụng gì để trang trí bánh?    | Truth: mấy cái hoa hồi                | Raw: hoa hoa hồi cho dễ thương hơn  | Span: hoa hồi cho dễ thương nữa nè k | EM_raw=0 EM_span=0 F1_raw=0.095 F1_span=0.160
[3/1062] Q: Bột ca cao được hòa tan bằng gì?         | Truth: nước ấm                        | Raw: 300ml nước cốt dừa với sữa đặc | Span: 300ml nước cốt dừa với sữa đặc | EM_raw=0 EM_span=0 F1_raw=0.051 F1_span=0.051
[4/1062] Q: Việc làm nóng lò trước khi nướng để làm  | Truth: tạo rễ tre cho bánh và để bánh | Raw: Tạo rèn cho bánh và để bánh đư | Span: Tạo rèn cho bánh và để bánh đư | EM_raw=0 EM_span=0 F1_raw=0.391 F1_span=0.391
[5/1062] Q: Nếu để hỗn hợp ca cao sôi